## Build a Simple LLM Application with LCEL

In this quickstart we'll see, how to build a simple LLM application with LangChain. This application will transalte text from English into another language. This is relatively simple LLM APPLICATION - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting. Still this is a gerat way to get started with LangChain - a lot of features can be built with just prompting and an LLM call!

After this, we'll have a high level overview of:
- Using language models
- Using PromptTemplates and OutputParsers
- Using Langchain Expression Language (LCEL) to chain components together
- Debugging and tracing your application using LangSmith
- Deploying your application to LangServe


In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import openai
openai.api_key=os.getenv("OPENAI_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")

In [4]:
!pip install openai


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from openai import OpenAI
client = OpenAI()

In [6]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Write a one sentence bedtime story about a unicorn."
)

print(response)

Response(id='resp_00ce0a31f59ad63c006ac537e1dce887d2a9c1a749a46cb588', created_at=1791309793.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_00ce0a31f59ad63c006ac537e3981887d2aa24545d33aa3188', content=[ResponseOutputText(annotations=[], text='Under a sky sprinkled with twinkling stars, a gentle unicorn named Luna spread her shimmering wings to sprinkle dreams of magic and joy over the sleeping world.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1791309796.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='in_memory', reasoning=Reasoning(effort=None, generate_summary=None, summary=None, context=None),

In [7]:
response.output_text

'Under a sky sprinkled with twinkling stars, a gentle unicorn named Luna spread her shimmering wings to sprinkle dreams of magic and joy over the sleeping world.'

In [8]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

In [13]:
model = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000210C3C50B00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000210C3C53440>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
from langchain_core.messages import HumanMessage, SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to Hindi"),
    HumanMessage(content="Hello, How are you?")
]

result=model.invoke(messages)

In [17]:
print(result)

content='नमस्ते, आप कैसे हैं?' additional_kwargs={'reasoning_content': 'The user wants translation from English to Hindi. The phrase "Hello, How are you?" Should translate to Hindi. Probably "नमस्ते, आप कैसे हैं?" or "हैलो, आप कैसे हैं?" Use standard. Provide translation.'} response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 87, 'total_tokens': 156, 'completion_time': 0.144518976, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.01324074, 'prompt_tokens_details': None, 'queue_time': 0.411308726, 'total_time': 0.157759716}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1126c-6525-7c92-8b64-ef93bf7c6b55-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 87, 'output_tokens': 69, 'total_tokens': 156, 'output_token_details': {'reasoning': 51}}


In [18]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [19]:
parser.invoke(result)

'नमस्ते, आप कैसे हैं?'

In [20]:
### Using LCEL - chain the component
chain = model | parser

In [21]:
chain.invoke(messages)

'नमस्ते, आप कैसे हैं?'

In [23]:
### Prompt Template
from langchain_core.prompts import ChatPromptTemplate

generic_template="Transalte the following into {language} :"



In [31]:
prompt = ChatPromptTemplate(
    [
        ("system", generic_template),
        ("user", "{text}")
    ]
)

In [33]:
result = prompt.invoke({"language":"French","text":"Hello"})

In [34]:
result.to_messages()

[SystemMessage(content='Transalte the following into French :', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [36]:
chain = prompt| model | parser

In [45]:
chain.invoke({"language":"India(English)","text":"Hello"})

'Hello (commonly used in Indian English as well)'